In [ ]:
"""! pip install langchain_core
!pip install langchain_google_genai
!pip install faiss-cpu
! pip install langchain_community
!pip install pymupdf
!pip install -U langchain langchain-core langchain-community langchain-huggingface transformers torch sentence-transformers huggingface-hub nltk regex pypdf unstructured accelerate bitsandbytes"""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.tools import Tool
from langchain_google_genai import GoogleGenerativeAI,ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from typing import Annotated,TypedDict,Sequence
from langgraph.prebuilt import create_react_agent,chat_agent_executor
from langgraph.graph import END,state,StateGraph
from langgraph.graph.message import add_messages,BaseMessage

In [ ]:
#!pip install nbformat

In [ ]:
#!pip install gradio # key ====

In [ ]:
os.environ['GROQ_API_KEY']="HC"

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # correct Groq model name
    temperature=0
)



In [ ]:
import os
from typing import Annotated, TypedDict, Sequence

# Langchain imports
from langchain_community.document_loaders import PyMuPDFLoader,NotebookLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LangGraph
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.prebuilt import ToolNode


# =========================
# API KEY
# =========================
os.environ["GOOGLE_API_KEY"] = "AIzaSyDHrlZcU"

### this is for mental tension ok !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
loaders= PyMuPDFLoader("/content/MHGuidebook-EBookDownload.pdf")
loadeds =loaders.load()
splits=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n","\n\n",",","."]
)
docs=splits.split_documents(loadeds)
e=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=FAISS.from_documents(docs,embedding=e)
def formatter(docs):
  return"\n".join(doc.page_content for doc in docs)
retri=vector_store.as_retriever(search_kwargs={"k":5},search_type="mmr")
##################################################################


## Loading Insurance Price Prediction model into it now

loading=NotebookLoader("/content/insurance_project (2).ipynb",include_outputs=True)
loaded=loading.load()
split=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=[",",".","\n","\n\n"]
)
docss=split.split_documents(loaded)
e=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_stores=FAISS.from_documents(docss,embedding=e)
retriever=vector_stores.as_retriever(search_kwargs={"k":5},search_type="mmr")

################

# =========================
# LLM
# =========================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)


# =========================
# PROMPT 1
# =========================
prompt = PromptTemplate.from_template(
"""
You are an assistant for medical help.

Help the patient in the best possible way.
The patient might be depressed, respond positively.

Question:
{question}
{context} use the folowing to answer in a better way ok
"""
)

tool_chain1 = (
    {
        "context": retri | formatter,
        "question": RunnablePassthrough(),
    }

    | prompt
    | llm
    | StrOutputParser()
)


# =========================
# PROMPT 2
# =========================
prompt1 = PromptTemplate.from_template(
"""
You are a medical information assistant specialized in explaining the side effects of medicines.

Your task is to:
- Explain possible side effects clearly
- Explain impact on body
- Mention common and rare side effects

Rules:
- Only explain side effects
- Do NOT prescribe medicine
- Advise consulting doctor
- Keep response concise

User Question:
{question}
"""
)

tool_chain2 = (
    {"question": RunnablePassthrough()}
    | prompt1
    | llm
    | StrOutputParser()
)


# =========================
# PROMPT 2
# =========================

prompt3=PromptTemplate.from_template(
    """
    you are helpful assistant this is the question use it it and understand it carefully{question}
    and the context for qustion is {context} the total will be about isurance price if it is being
    anwer properly Answer:

    """)
tool_chain3=(
    {
        "context":retriever|formatter,
        "question":RunnablePassthrough()
    }|prompt3|llm|StrOutputParser()
)




# =========================
# TOOLS
# =========================
tooler = Tool(
    name="medical_help",
    description="general medical help assistant",
    func=tool_chain1.invoke
)

tooler2 = Tool(
    name="medicine_side_effects",
    description="explains side effects of medicines",
    func=tool_chain2.invoke
)
tooler3=Tool(
    name="insurance_price_prediction",
    description="predicts insurance price and understand about insurance completely ",
    func=tool_chain3.invoke
)

tools = [tooler, tooler2,tooler3]

llm = llm.bind_tools(tools)# binding the tools with the llm


# =========================
# STATE
# =========================
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


# =========================
# LLM NODE
# =========================
def llm_node(state: AgentState):

    response = llm.invoke(state["messages"])

    return {
        "messages": state["messages"] + [response]
    }


# =========================
# TOOL NODE
# =========================
tool_node = ToolNode(tools)


# =========================
# CONDITION
# =========================




def should_continue(state: AgentState):

    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return END


# =========================
# GRAPH
# =========================
graph = StateGraph(AgentState)

graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("llm")

graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

graph.add_edge("tools", "llm")

agent = graph.compile()


# =========================
# ASK FUNCTION
# =========================
def ask_agent(question: str):

    response = agent.invoke(
        {
            "messages": [HumanMessage(content=question)]
        }
    )

    return response["messages"][-1].content

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#ask_agent("explain me the depression reasoon??")

In [ ]:
#AI

In [ ]:
import gradio as gr

def chat_with_agent(message, history):

    response = ask_agent(message)

    history.append((message, response))

    return history, history


css = """
body {
    background-color: #0f172a;
}

.gradio-container {
    background-color: #0f172a;
    color: white;
}

#chatbot {
    background-color: #111827;
    border-radius: 12px;
    border: 1px solid #2563eb;
}

textarea {
    background-color: #111827 !important;
    color: white !important;
    border: 1px solid #2563eb !important;
}

button {
    background-color: #2563eb !important;
    color: white !important;
    border-radius: 8px !important;
}

button:hover {
    background-color: #1d4ed8 !important;
}
"""


with gr.Blocks(css=css) as demo:

    gr.Markdown(
        """
        # 🏥 AI Medical Assistant
        Ask health-related questions and get AI guidance.
        """
    )

    chatbot = gr.Chatbot(elem_id="chatbot")

    msg = gr.Textbox(
        placeholder="Ask your medical question...",
        label="Your Question"
    )

    clear = gr.Button("Clear Chat")

    msg.submit(
        chat_with_agent,
        inputs=[msg, chatbot],
        outputs=[chatbot, chatbot]
    )

    clear.click(lambda: None, None, chatbot, queue=False)


demo.launch()

/tmp/ipykernel_161/1394245383.py:46: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as demo:
/tmp/ipykernel_161/1394245383.py:55: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_id="chatbot")
/tmp/ipykernel_161/1394245383.py:55: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(elem_id="chatbot")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c99b9775c565df185d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!python --version

Python 3.12.12
